In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("superstore.csv")

In [10]:
df = pd.DataFrame(data)

In [35]:
df.head()

,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,记录数,Order.Date,Order.ID,...,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,2011-01-07 00:00:00.000,CA-2011-130813,...,19,Consumer,2011-01-09 00:00:00.000,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,2011-01-21 00:00:00.000,CA-2011-148614,...,19,Consumer,2011-01-26 00:00:00.000,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,21,Consumer,2011-08-09 00:00:00.000,Standard Class,1.81,California,Paper,2011,North America,32
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,111,Consumer,2011-08-09 00:00:00.000,Standard Class,4.59,California,Paper,2011,North America,32
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,1,2011-09-29 00:00:00.000,CA-2011-146969,...,6,Consumer,2011-10-03 00:00:00.000,Standard Class,1.32,California,Paper,2011,North America,40


#### Nettoyage des chaines de caractères (suppression des espaces invisibles)

In [34]:
df["Customer.ID"] = df['Customer.ID'].str.strip()
df['Customer.Name'] = df['Customer.Name'].str.strip()
df["Product.Name"] = df['Product.Name'].str.strip()

#### Standardisation des dates . On force le format date pour pouvoir faire des analyses chronologiques

In [37]:
df["Order.Date"] = pd.to_datetime(df["Order.Date"], errors='coerce')
df['Ship.Date'] = pd.to_datetime(df["Ship.Date"],errors= "coerce")

#### Création de 3 tables distinctes selon le modèle relationnel et supprimons les doublons

In [ ]:
customers = df[["Customer.ID","Customer.Name","Segment","Country","City","State","Region"]].copy()
customers.drop_duplicates("Customer.ID",inplace=True)


In [25]:
products = df[["Product.ID","Category","Sub.Category","Product.Name"]].copy()
products.drop_duplicates("Product.ID",inplace=True)

In [26]:
transactions = df[["Order.ID","Order.Date","Ship.Date","Ship.Mode","Customer.ID","Product.ID","Sales","Quantity","Discount","Profit"]].copy()
transactions.drop_duplicates("Order.ID",inplace=True)

#### Création du "Flag" d'audit interne (Alerte sur les grosses remises à perte)
lorsque le profit est négatif + la remise dépasse 40%

In [ ]:
transactions['Red.Flag'] = (transactions['Profit'] < 0) & (transactions['Discount'] > 0.4)
transactions[transactions["Red.Flag"]==True]

#### Calcul du coût de revient et vérifions s'il ya des transactions avec un coût de revient négatif ou CA négatif

In [51]:
transactions["Cout de Revient"] = transactions["Sales"] - transactions["Profit"]
if transactions[transactions['Cout de Revient'] < 0].empty:
    print("Il n'y a pas de transactions avec un coût de revient négatif.")
else:
    print("Il Y a des transactions avec un coût de revient négatif.")

Il n'y a pas de transactions avec un coût de revient négatif.


In [50]:
if transactions[transactions['Sales'] < 0].empty:
    print("Il n'y a pas de transactions avec des ventes négatives.")
else:
    print("Il Y a des transactions avec des ventes négatives.")



Il n'y a pas de transactions avec des ventes négatives.


#### Vérification Logique des Frais de Port 
Il arrive que le coût de livraison (Shipping Cost) soit supérieur au montant total de la commande (Sales). C'est une anomalie logistique grave qu'un contrôleur de gestion doit immédiatement signaler. On fait d'abord le text logistique puis on ajoute toutes les exceptions dans une table . Ensuite on ajoute une colonne motif pour spécifier la raison de l'anomalie . Finalement on exporte les exceptions en EXCEL.

In [ ]:
anomalie_logistique= df["Shipping.Cost"]> df["Sales"]
exceptions = df[anomalie_logistique].copy()
exceptions['Motif de l\'anomalie'] =  "Coût de livraison supérieur au montant de la vente"
df[~anomalie_logistique]
print(f"Nettoyage terminé : {len(exceptions)} anomalie(s) isolée(s) dans le rapport d'exceptions.")

#### Concentration de chiffre d'affaire : Test de la loi 80/20

In [ ]:
total_CA = transactions["Sales"].sum()
CA_par_region = df.groupby("Region")["Sales"].sum().reset_index() # The reset_index() is used to convert the Series back to a DataFrame and copy() is used to avoid SettingWithCopyWarning so when u want to add a new column it wont add new rows
CA_par_region["Part de marché"] = (CA_par_region["Sales"] / total_CA) * 100
CA_par_region.sort_values("Part de marché", ascending=False,inplace=True)
CA_par_region["Part de marché Cumulée"] = CA_par_region["Part de marché"].cumsum().round(2) # cumsum() is used to calculate the cumulative sum of the "Part de marché" column, which gives us the cumulative market share as we go down the sorted list of regions.
top_regions = CA_par_region[CA_par_region['Part de marché Cumulée'] <= 80]['Region'].tolist()
if len(top_regions) > 0:
    regions_texte = ", ".join(top_regions)
    value = "⚠️ Alerte Risque de Concentration : Près de 80% de notre chiffre d'affaires repose uniquement sur ces régions : {regions_texte}."
    print(value)
else:
    print("La première région représente à elle seule plus de 80% du CA !")
CA_par_region["Risque de Concentration"] =value


⚠️ Alerte Risque de Concentration : Près de 80% de notre chiffre d'affaires repose uniquement sur ces régions : {regions_texte}.


#### Exportation des données nettoyées

In [135]:

transactions.to_csv("transactions.csv", index=False)
customers.to_csv("customers.csv", index=False)
products.to_csv("products.csv", index=False)
exceptions.to_excel("exceptions_logistiques.xlsx",index=False)
CA_par_region.to_excel("CA_par_region.xlsx", index=False)
print("Nettoyage et exportation terminés avec succès !")

Nettoyage et exportation terminés avec succès !


## Analyse des écarts et requêtage (SQL)

In [136]:
import sqlite3

In [167]:
connection = sqlite3.connect("audit_superstore.db")
customers.to_sql("customers",connection,if_exists="replace",index=0)
products.to_sql("products",connection,if_exists="replace",index=0)
transactions.to_sql("transactions",connection,if_exists="replace",index=0)
exceptions.to_sql("exceptions_logistiques",connection,if_exists="replace",index=0)
CA_par_region.to_sql("CA_par_region",connection,if_exists="replace",index=0)
print("Exportation vers la base de données SQLite terminée avec succès !")

Exportation vers la base de données SQLite terminée avec succès !


### AUDIT DE RENTABILITE

In [177]:
# Requête SQL corrigée avec le point
query_profitabilité = """
SELECT 
    p.Category,
    p."Sub.Category",
    SUM(t.Sales) as Chiffre_Affaires_Total,
    SUM(t.Profit) as Profit_Net_Total,
    AVG(t.Discount) as Remise_Moyenne
FROM Transactions t
JOIN Products p ON t."Product.ID" = p."Product.ID"
GROUP BY p.Category, p."Sub.Category"
HAVING Profit_Net_Total < 0
ORDER BY Profit_Net_Total ASC
LIMIT 5;
"""

top_pertes = pd.read_sql_query(query_profitabilité, connection)
print(top_pertes)

pertes = pd.read_sql_query(query_profitabilité,connection)
print("Top 5 des catégories et sous-catégories les plus déficitaires :")
display(pertes)

     Category Sub.Category  Chiffre_Affaires_Total  Profit_Net_Total  \
0   Furniture       Tables                  347103       -66245.6352   
1  Technology     Machines                  295586        -6892.6549   

   Remise_Moyenne  
0        0.344339  
1        0.231648  
Top 5 des catégories et sous-catégories les plus déficitaires :


,Category,Sub.Category,Chiffre_Affaires_Total,Profit_Net_Total,Remise_Moyenne
0,Furniture,Tables,347103,-66245.6352,0.344339
1,Technology,Machines,295586,-6892.6549,0.231648


#### L'Audit du "Red Flag" des Remises (Requête SQL n°2)

In [184]:
# Requête SQL : Localisation des remises abusives
query_red_flags = """
SELECT 
    c.Region,
    c.Segment,
    COUNT(t."Order.ID") as Nombre_Transactions_Suspectes,
    SUM(t.Profit) as Impact_Financier_Negatif
FROM Transactions t
JOIN Customers c ON t."Customer.ID" = c."Customer.ID"
WHERE t."Red.Flag" = True
GROUP BY c.Region, c.Segment
ORDER BY Impact_Financier_Negatif ASC
LIMIT 5;
"""

# Afficher les résultats
fraudes_potentielles = pd.read_sql_query(query_red_flags, connection)
print(" ZONES GÉOGRAPHIQUES À RISQUE (Abus de remises) :")
print(fraudes_potentielles)

 ZONES GÉOGRAPHIQUES À RISQUE (Abus de remises) :
    Region    Segment  Nombre_Transactions_Suspectes  Impact_Financier_Negatif
0     EMEA   Consumer                            377              -35506.75200
1   Africa   Consumer                            239              -29505.39300
2     West   Consumer                            225              -27301.22240
3  Central   Consumer                            218              -26624.77384
4     EMEA  Corporate                            228              -26431.46700


### Requête SQL : Évolution du Chiffre d'Affaires et de la Rentabilité (Year-over-Year)


In [ ]:
query_yoy = """
WITH Ventes_Annuelles AS (
    -- Étape 1 : On agresse les données par année avec strftime au lieu de YEAR()
    SELECT 
        strftime('%Y', t."Order.Date") AS Annee,
        SUM(t.Sales) AS CA_Total,
        SUM(t.Profit) AS Profit_Total
    FROM Transactions t
    WHERE t."Order.Date"   IS NOT NULL
    GROUP BY Annee
)
-- Étape 2 : On compare l'année en cours avec l'année précédente grâce à LAG()
SELECT 
    Annee,
    CA_Total AS CA_Annee_En_Cours,
    LAG(CA_Total) OVER(ORDER BY Annee) AS CA_Annee_Precedente,
    
    -- Calcul de l'évolution en % du CA
    ROUND(((CA_Total - LAG(CA_Total) OVER(ORDER BY Annee)) / LAG(CA_Total) OVER(ORDER BY Annee)) * 100, 2) AS Croissance_CA_Pourcentage,
    
    Profit_Total AS Profit_Annee_En_Cours,
    
    -- Calcul de l'évolution en % du Profit
    ROUND(((Profit_Total - LAG(Profit_Total) OVER(ORDER BY Annee)) / LAG(Profit_Total) OVER(ORDER BY Annee)) * 100, 2) AS Croissance_Profit_Pourcentage

FROM Ventes_Annuelles;
"""

# Lecture et affichage propre du résultat
evolution_yoy = pd.read_sql_query(query_yoy, connection)

# Rendu visuel "Rapport Financier"
def alerte_croissance(val):
    if pd.isna(val): # Pour la première année qui n'a pas de précédent
        return ''
    couleur = 'red' if val < 0 else 'green'
    return f'color: {couleur}; font-weight: bold'

rapport_yoy = evolution_yoy.style.format({
    'CA_Annee_En_Cours': '{:,.0f} $',
    'CA_Annee_Precedente': '{:,.0f} $',
    'Croissance_CA_Pourcentage': '{:+.2f} %',
    'Profit_Annee_En_Cours': '{:,.0f} $',
    'Croissance_Profit_Pourcentage': '{:+.2f} %'
}).map(alerte_croissance, subset=['Croissance_CA_Pourcentage', 'Croissance_Profit_Pourcentage'])

display(rapport_yoy)

,Annee,CA_Annee_En_Cours,CA_Annee_Precedente,Croissance_CA_Pourcentage,Profit_Annee_En_Cours,Croissance_Profit_Pourcentage
0,2011,"1,009,498 $",nan $,+nan %,"83,043 $",+nan %
1,2012,"1,179,253 $","1,009,498 $",+0.00 %,"98,149 $",+18.19 %
2,2013,"1,536,033 $","1,179,253 $",+0.00 %,"139,363 $",+41.99 %
3,2014,"1,907,972 $","1,536,033 $",+0.00 %,"158,371 $",+13.64 %


### Le Profilage des "Clients Toxiques" vs "Clients VIP"

In [202]:
# Requête SQL : Profilage des Clients (Toxiques vs VIP)
query_clients = """
WITH Profil_Client AS (
    -- Étape 1 : On aggrège les performances par client
    SELECT 
        c.'Customer.Name' AS Nom_Client,
        SUM(t.Sales) AS CA_Total,
        SUM(t.Profit) AS Profit_Total,
        AVG(t.Discount) AS Remise_Moyenne,
        COUNT(t.'Order.ID') AS Nombre_Commandes
    FROM Transactions t
    JOIN Customers c ON t."Customer.ID" = c."Customer.ID"
    GROUP BY c."Customer.ID", c.'Customer.Name'
)
-- Étape 2 : On classifie les clients avec un CASE WHEN
SELECT 
    Nom_Client,
    CA_Total,
    Profit_Total,
    Remise_Moyenne,
    Nombre_Commandes,
    CASE
        WHEN CA_Total > 5000 AND Profit_Total < 0 THEN '🛑 Client Toxique'
        WHEN CA_Total > 5000 AND Profit_Total > 1000 THEN '⭐ Client VIP'
        ELSE 'Standard'
    END AS Segment_Audit
FROM Profil_Client
-- On filtre pour ne garder que les extrêmes dans notre rapport
WHERE Segment_Audit IN ('🛑 Client Toxique', '⭐ Client VIP')
ORDER BY Profit_Total ASC;
"""

# Lecture des résultats
segmentation_clients = pd.read_sql_query(query_clients, connection)

# Rendu visuel "Contrôle de Gestion"
def style_segmentation(row):
    # Si le segment est Toxique, on met la ligne en rouge léger, sinon en vert léger
    if row['Segment_Audit'] == '🛑 Client Toxique':
        return ['background-color: #ffe6e6; color: #cc0000; font-weight: bold'] * len(row)
    elif row['Segment_Audit'] == '⭐ Client VIP':
        return ['background-color: #e6ffe6; color: #006600'] * len(row)
    return [''] * len(row)

rapport_clients = segmentation_clients.style.format({
    'CA_Total': '{:,.0f} $',
    'Profit_Total': '{:,.0f} $',
    'Remise_Moyenne': '{:.1%}'
}).apply(style_segmentation, axis=1) # On applique la couleur sur toute la ligne

display(rapport_clients)

,Nom_Client,CA_Total,Profit_Total,Remise_Moyenne,Nombre_Commandes,Segment_Audit
0,Cindy Stewart,"5,161 $","-6,340 $",15.0%,6,🛑 Client Toxique
1,Grant Thornton,"8,042 $","-3,827 $",23.3%,3,🛑 Client Toxique
2,Sean Miller,"23,282 $","-1,855 $",28.0%,5,🛑 Client Toxique
3,Carl Ludwig,"6,235 $","-1,305 $",7.2%,14,🛑 Client Toxique
4,John Lee,"5,259 $","-1,167 $",12.7%,11,🛑 Client Toxique
5,Joseph Holt,"6,536 $","-1,153 $",13.3%,6,🛑 Client Toxique
6,Mike Pelletier,"5,470 $",-723 $,28.7%,13,🛑 Client Toxique
7,Hunter Glantz,"5,123 $",-609 $,20.4%,9,🛑 Client Toxique
8,Stuart Van,"5,877 $",-377 $,6.7%,7,🛑 Client Toxique
9,Corey Roper,"6,317 $",-340 $,19.7%,12,🛑 Client Toxique


#### EXPORTONS TOUT CA EN EXCEL

In [203]:
with pd.ExcelWriter('Audit_Reporting_Superstore.xlsx') as writer:
    # 1. Les données sources nettoyées (Pour Power BI plus tard)
    transactions.to_excel(writer, sheet_name='Transactions_Clean', index=False)
    customers.to_excel(writer, sheet_name='Customers', index=False)
    products.to_excel(writer, sheet_name='Products', index=False)
    
    # 2. Les constats d'audit (Tes requêtes SQL)
    top_pertes.to_excel(writer, sheet_name='Diagnostic_Pertes', index=False)
    segmentation_clients.to_excel(writer, sheet_name='Audit_Clients', index=False)

print("Fichier Excel 'Audit_Reporting_Superstore.xlsx' généré avec succès !.")

Fichier Excel 'Audit_Reporting_Superstore.xlsx' généré avec succès !.
